# MNIST — remote-gpu example
Trains a small MLP. Runs identically locally and on Kaggle (paths are swapped by remote-gpu).

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim

INPUT_PATH = "./input"
OUTPUT_PATH = "./output"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

device: cpu


In [2]:
data = torch.load(os.path.join(INPUT_PATH, "mnist.pt"), weights_only=True)

X_train = data["X_train"].float().div_(255).unsqueeze(1)  # (N, 1, 28, 28)
y_train = data["y_train"].long()
X_test = data["X_test"].float().div_(255).unsqueeze(1)
y_test = data["y_test"].long()

train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_test, y_test), batch_size=256)

print(f"train: {len(X_train)}  test: {len(X_test)}")

train: 60000  test: 10000


In [3]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, 10),
).to(device)

print(model)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=256, bias=True)
  (2): ReLU()
  (3): Dropout(p=0.2, inplace=False)
  (4): Linear(in_features=256, out_features=10, bias=True)
)


In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(3):
    model.train()
    total_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"epoch {epoch + 1}: loss {total_loss / len(train_loader):.4f}")

epoch 1: loss 0.3849
epoch 2: loss 0.1743
epoch 3: loss 0.1233


In [5]:
model.eval()
correct = 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()

print(f"test accuracy: {correct / len(X_test):.4f}")

os.makedirs(OUTPUT_PATH, exist_ok=True)
model_file = os.path.join(OUTPUT_PATH, "mnist_model.pt")
torch.save(model.state_dict(), model_file)
print(f"saved: {model_file}")

test accuracy: 0.9719
saved: ./output\mnist_model.pt
